# Lakelink Fiber Provable Resilience: @data-analyst v4 candidate writers

**Owner:** @data-analyst. **Task:** P48-DA-V4-WRITERS-CHECKIN (CI gate `test_v4_candidate_writer_contains_literal_name`, BUG-QA-085).

**Objects written (only):**

| Object | Type | Origin task | Source |
| --- | --- | --- | --- |
| `cdm_tmforum.provable_resilience_intelligence.revenue_at_risk_v4_candidate` | Metric view (YAML 1.1) | BUG-QA-069 | frozen v3 `resilience_score_snapshot` + `service_inventory` + `customer` |

The DDL below was captured from the live object with `SHOW CREATE TABLE` on 2026-10-27, so this notebook reproduces exactly what is deployed.

**Freeze rules (DEC-PRI-V4-SHADOW-BUILD):**
* `write_enabled` defaults to `false`. With the default, the notebook only reads: it checks that the live definition matches the checked-in DDL and that the governed KPIs reproduce.
* Set `write_enabled=true` only in a @pm-approved write window. The write cell asserts the target ends in `_v4_candidate`.
* At v4 cutover (P30-DA-GENIE-MRC-PROTECTED-CUTOVER), repoint the `source` query to `resilience_score_snapshot_v4_candidate` here first, then re-run with `write_enabled=true`.

In [0]:
CATALOG_SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
RAR_MV = f"{CATALOG_SCHEMA}.revenue_at_risk_v4_candidate"

dbutils.widgets.dropdown("write_enabled", "false", ["false", "true"], "Write enabled (@pm write window only)")
WRITE_ENABLED = dbutils.widgets.get("write_enabled") == "true"

# Freeze guard: this notebook may only write *_v4_candidate objects.
assert RAR_MV.endswith("_v4_candidate"), f"Refusing to write a non-candidate object: {RAR_MV}"

# Governed KPI baseline published by BUG-QA-069 (frozen v3 snapshot, data v22).
RAR_BASELINE = {"critical_service_count": 848, "at_risk_service_count": 2932, "total_revenue_at_risk": 48445093}
print(f"target={RAR_MV} write_enabled={WRITE_ENABLED}")

In [0]:
# Captured from the live object (SHOW CREATE TABLE, 2026-10-27). BUG-QA-069 governed bands:
# critical = overall_score < 40, at_risk = 40 <= overall_score < 60 (was score < 70 in the pipeline MV).
RAR_MV_DDL = r"""
CREATE OR REPLACE VIEW cdm_tmforum.provable_resilience_intelligence.revenue_at_risk_v4_candidate (
  customer_id COMMENT 'Lakelink Fiber customer identifier (FK to customer.customer_id)',
  customer_name COMMENT 'Customer legal/display name',
  customer_tier COMMENT 'Customer tier: platinum, gold, silver',
  industry COMMENT 'Customer industry vertical',
  region COMMENT 'Customer geographic region',
  customer_sla_tier COMMENT 'Customer SLA tier: premium_plus, premium, standard',
  service_type COMMENT 'Service type: wavelength, ethernet, dark_fiber, ip_vpn, managed_security',
  risk_level COMMENT 'Governed risk band: healthy (>=80), moderate (60-79), at_risk (40-59), critical (<40)',
  service_count COMMENT 'Distinct scored active services',
  customer_count COMMENT 'Distinct customers with scored active services',
  avg_resilience_score COMMENT 'Average composite resilience score (0-100, higher is better)',
  min_resilience_score COMMENT 'Lowest composite resilience score in the group',
  critical_service_count COMMENT 'Services with overall_score < 40 (governed critical KPI)',
  at_risk_service_count COMMENT 'Services with 40 <= overall_score < 60 (governed at_risk KPI; excludes critical). BUG-QA-069 fix: replaces pipeline score<70 definition',
  total_revenue_at_risk COMMENT 'Total monthly revenue at risk, USD (mrc_usd * (1 - overall_score/100))',
  critical_revenue_at_risk COMMENT 'Monthly revenue at risk from critical services (<40), USD',
  at_risk_revenue_at_risk COMMENT 'Monthly revenue at risk from at_risk services (40-59), USD',
  total_active_hazards COMMENT 'Sum of active hazards across services (service-hazard pairs, not distinct hazards)',
  total_shared_srlgs COMMENT 'Sum of shared SRLGs between working and protect paths across services')
COMMENT 'SHADOW v4 candidate (BUG-QA-069). Lakelink Fiber revenue at risk by customer, governed replacement for the pipeline gold_revenue_at_risk MV. Grain of source: one row per scored active service (resilience_score_snapshot) joined to service_inventory and customer. Fixes at_risk_services: governed KPI at_risk = 40 <= overall_score < 60 (was score < 70, which double-counted critical). critical = overall_score < 40. Sourced from the frozen v3 snapshot (data v22) so the fix is verifiable; repoint to resilience_score_snapshot_v4_candidate at v4 cutover.'
WITH METRICS
LANGUAGE YAML
AS
$$
version: 1.1

source: |-
  SELECT
    rss.service_id,
    rss.overall_score,
    rss.revenue_at_risk_usd,
    rss.active_hazards,
    rss.srlg_count,
    si.service_type,
    c.customer_id,
    c.customer_name,
    c.tier AS customer_tier,
    c.industry,
    c.region,
    c.sla_tier AS customer_sla_tier,
    CASE
      WHEN rss.overall_score >= 80 THEN 'healthy'
      WHEN rss.overall_score >= 60 THEN 'moderate'
      WHEN rss.overall_score >= 40 THEN 'at_risk'
      ELSE 'critical'
    END AS risk_level
  FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot rss
  JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si
    ON rss.service_id = si.service_id
  JOIN cdm_tmforum.provable_resilience_intelligence.customer c
    ON si.customer_id = c.customer_id
  WHERE si.status = 'active'

comment: "SHADOW v4 candidate (BUG-QA-069). Lakelink Fiber revenue at risk by customer,\
  \ governed replacement for the pipeline gold_revenue_at_risk MV. Grain of source:\
  \ one row per scored active service (resilience_score_snapshot) joined to service_inventory\
  \ and customer. Fixes at_risk_services: governed KPI at_risk = 40 <= overall_score\
  \ < 60 (was score < 70, which double-counted critical). critical = overall_score\
  \ < 40. Sourced from the frozen v3 snapshot (data v22) so the fix is verifiable;\
  \ repoint to resilience_score_snapshot_v4_candidate at v4 cutover."

dimensions:
  - name: customer_id
    expr: customer_id
    comment: Lakelink Fiber customer identifier (FK to customer.customer_id)
    display_name: Customer ID

  - name: customer_name
    expr: customer_name
    comment: Customer legal/display name
    display_name: Customer Name

  - name: customer_tier
    expr: customer_tier
    comment: "Customer tier: platinum, gold, silver"
    display_name: Customer Tier

  - name: industry
    expr: industry
    comment: Customer industry vertical
    display_name: Industry

  - name: region
    expr: region
    comment: Customer geographic region
    display_name: Region

  - name: customer_sla_tier
    expr: customer_sla_tier
    comment: "Customer SLA tier: premium_plus, premium, standard"
    display_name: Customer SLA Tier

  - name: service_type
    expr: service_type
    comment: "Service type: wavelength, ethernet, dark_fiber, ip_vpn, managed_security"
    display_name: Service Type

  - name: risk_level
    expr: risk_level
    comment: "Governed risk band: healthy (>=80), moderate (60-79), at_risk (40-59),\
      \ critical (<40)"
    display_name: Risk Level

measures:
  - name: service_count
    expr: COUNT(DISTINCT service_id)
    comment: Distinct scored active services
    display_name: Services

  - name: customer_count
    expr: COUNT(DISTINCT customer_id)
    comment: Distinct customers with scored active services
    display_name: Customers

  - name: avg_resilience_score
    expr: "ROUND(AVG(overall_score), 1)"
    comment: "Average composite resilience score (0-100, higher is better)"
    display_name: Avg Resilience Score

  - name: min_resilience_score
    expr: "ROUND(MIN(overall_score), 1)"
    comment: Lowest composite resilience score in the group
    display_name: Min Resilience Score

  - name: critical_service_count
    expr: COUNT(DISTINCT service_id) FILTER (WHERE overall_score < 40)
    comment: Services with overall_score < 40 (governed critical KPI)
    display_name: Critical Services

  - name: at_risk_service_count
    expr: COUNT(DISTINCT service_id) FILTER (WHERE overall_score >= 40 AND overall_score
      < 60)
    comment: "Services with 40 <= overall_score < 60 (governed at_risk KPI; excludes\
      \ critical). BUG-QA-069 fix: replaces pipeline score<70 definition"
    display_name: At-Risk Services
    synonyms:
      - at risk services
      - at_risk_services

  - name: total_revenue_at_risk
    expr: SUM(revenue_at_risk_usd)
    comment: "Total monthly revenue at risk, USD (mrc_usd * (1 - overall_score/100))"
    display_name: Total Revenue at Risk
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: critical_revenue_at_risk
    expr: SUM(revenue_at_risk_usd) FILTER (WHERE overall_score < 40)
    comment: "Monthly revenue at risk from critical services (<40), USD"
    display_name: Critical Revenue at Risk
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: at_risk_revenue_at_risk
    expr: SUM(revenue_at_risk_usd) FILTER (WHERE overall_score >= 40 AND overall_score
      < 60)
    comment: "Monthly revenue at risk from at_risk services (40-59), USD"
    display_name: At-Risk Band Revenue at Risk
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact

  - name: total_active_hazards
    expr: SUM(active_hazards)
    comment: "Sum of active hazards across services (service-hazard pairs, not distinct\
      \ hazards)"
    display_name: Active Hazard Exposures

  - name: total_shared_srlgs
    expr: SUM(srlg_count)
    comment: Sum of shared SRLGs between working and protect paths across services
    display_name: Shared SRLGs
$$
""".strip()

In [0]:
import re

def _norm(sql: str) -> str:
    """Drop the CREATE prefix and collapse whitespace so only the body is compared."""
    body = re.sub(r"^\s*CREATE\s+(OR\s+REPLACE\s+)?VIEW\s+", "", sql.strip(), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", body).strip()

live_ddl = spark.sql(f"SHOW CREATE TABLE {RAR_MV}").first()[0]
DDL_MATCHES_LIVE = _norm(live_ddl) == _norm(RAR_MV_DDL)
print(f"live definition matches checked-in DDL: {DDL_MATCHES_LIVE}")

In [0]:
kpis = spark.sql(f"""
  SELECT MEASURE(service_count) AS service_count,
         MEASURE(critical_service_count) AS critical_service_count,
         MEASURE(at_risk_service_count) AS at_risk_service_count,
         ROUND(MEASURE(total_revenue_at_risk)) AS total_revenue_at_risk
  FROM {RAR_MV}""").first().asDict()

for k, expected in RAR_BASELINE.items():
    print(f"{k}: live={kpis[k]} baseline={expected} match={int(kpis[k]) == expected}")
print(f"service_count: {kpis['service_count']}")

In [0]:
if WRITE_ENABLED:
    assert RAR_MV.endswith("_v4_candidate")
    spark.sql(RAR_MV_DDL)
    print(f"wrote {RAR_MV}")
else:
    print("write_enabled=false: no write. Set it to true only in a @pm-approved write window.")